# Experiment 7: Perceptron and Forward Propagation
**Course:** CSA4703 – Deep Learning  
**Scenario:** A bank screens small loan applications using credit score and income-to-debt ratio. The notebook also tests the XOR pattern.

## Aim
Implement a single-layer perceptron using NumPy, train it on 100 loan applications, visualise its decision boundary, demonstrate why a single perceptron cannot learn XOR, and compare step, sigmoid and ReLU activations.

**Libraries:** NumPy, Matplotlib. The data is generated reproducibly for this lab exercise.

## 1. Perceptron model
A perceptron calculates the weighted sum \(z = w_1x_1 + w_2x_2 + b\), then applies a step activation. If \(z \geq 0\), the prediction is Approved (1); otherwise it is Rejected (0). The learning rule updates weights using \(w \leftarrow w + \eta(y-\hat y)x\) and \(b \leftarrow b + \eta(y-\hat y)\).

In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)
print('Libraries imported successfully.')

Libraries imported successfully.


## 2. Create 100 loan applications and train the perceptron
Feature 1 is a normalised credit score and Feature 2 is a normalised income-to-debt score. Labels are generated from a linear rule so the loan data is linearly separable.

In [2]:
# Generate 100 reproducible, linearly separable loan applications
rng = np.random.default_rng(7)
X = rng.uniform(-1, 1, size=(100, 2))
y = (1.2 * X[:, 0] - 0.9 * X[:, 1] + 0.1 > 0).astype(int)

def step_activation(z):
    return (z >= 0).astype(int)

weights = np.zeros(2)
bias = 0.0
learning_rate = 0.1
max_epochs = 1000
errors_per_epoch = []

for epoch in range(max_epochs):
    errors = 0
    for xi, target in zip(X, y):
        prediction = int(np.dot(xi, weights) + bias >= 0)
        error = target - prediction
        if error != 0:
            weights += learning_rate * error * xi
            bias += learning_rate * error
            errors += 1
    errors_per_epoch.append(errors)
    if errors == 0:
        break

predictions = step_activation(X @ weights + bias)
accuracy = np.mean(predictions == y)
print('Dataset shape:', X.shape)
print('Class counts (Rejected=0, Approved=1):', np.bincount(y))
print(f'Converged after {epoch + 1} epochs.')
print('Final weights:', np.round(weights, 4))
print(f'Final bias: {bias:.4f}')
print(f'Training accuracy: {accuracy * 100:.2f}%')

Dataset shape: (100, 2)
Class counts (Rejected=0, Approved=1): [47 53]
Converged after 107 epochs.
Final weights: [ 1.1205 -0.8279]
Final bias: 0.1000
Training accuracy: 100.00%


In [3]:
# Plot training errors and the learned decision boundary
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(range(1, len(errors_per_epoch) + 1), errors_per_epoch, marker='o')
axes[0].set_title('Perceptron errors per epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Number of misclassified samples')
axes[0].grid(True)

axes[1].scatter(X[y == 0, 0], X[y == 0, 1], label='Rejected (0)', marker='x')
axes[1].scatter(X[y == 1, 0], X[y == 1, 1], label='Approved (1)', marker='o')
x_line = np.linspace(-1, 1, 200)
if abs(weights[1]) > 1e-12:
    y_line = -(weights[0] * x_line + bias) / weights[1]
    axes[1].plot(x_line, y_line, 'k--', label='Decision boundary')
axes[1].set_xlim(-1.05, 1.05)
axes[1].set_ylim(-1.05, 1.05)
axes[1].set_xlabel('Normalised credit score')
axes[1].set_ylabel('Normalised income-to-debt score')
axes[1].set_title('Loan approval decision boundary')
axes[1].legend()
axes[1].grid(True)
plt.tight_layout()
plt.show()

## 3. Train the perceptron on XOR data
XOR outputs 1 when its two inputs are different and 0 when they are the same. This pattern is not linearly separable, so one straight decision boundary cannot classify all four points correctly.

In [4]:
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0])
w_xor = np.zeros(2)
b_xor = 0.0

for _ in range(20):
    for xi, target in zip(X_xor, y_xor):
        pred = int(np.dot(xi, w_xor) + b_xor >= 0)
        error = target - pred
        w_xor += 0.1 * error * xi
        b_xor += 0.1 * error

xor_predictions = step_activation(X_xor @ w_xor + b_xor)
xor_accuracy = np.mean(xor_predictions == y_xor)
print('XOR targets:', y_xor)
print('Perceptron predictions:', xor_predictions)
print(f'XOR accuracy: {xor_accuracy * 100:.2f}%')
print('Result: A single-layer perceptron cannot learn XOR (not linearly separable).')

plt.figure(figsize=(5, 4))
for point, label in zip(X_xor, y_xor):
    plt.scatter(point[0], point[1], s=140, marker='o' if label else 'x', label=f'Output {label}')
plt.title('XOR dataset: no single straight line separates the classes')
plt.xlabel('Input 1'); plt.ylabel('Input 2')
plt.xticks([0, 1]); plt.yticks([0, 1]); plt.grid(True)
plt.show()

XOR targets: [0 1 1 0]
Perceptron predictions: [1 1 0 0]
XOR accuracy: 50.00%
Result: A single-layer perceptron cannot learn XOR (not linearly separable).


## 4. Compare step, sigmoid and ReLU activations
The following cell applies three activation functions to the same input values. Step produces class-like outputs, sigmoid maps values to the range 0–1, and ReLU replaces negative values with zero.

In [5]:
z = np.array([-2., -1., 0., 1., 2.])
sigmoid = 1 / (1 + np.exp(-z))
relu = np.maximum(0, z)
step = step_activation(z)
print('Input values:', z)
print('Step:', step)
print('Sigmoid:', np.round(sigmoid, 4))
print('ReLU:', relu)

plt.figure(figsize=(8, 5))
plt.plot(z, step, marker='o', label='Step')
plt.plot(z, sigmoid, marker='o', label='Sigmoid')
plt.plot(z, relu, marker='o', label='ReLU')
plt.title('Activation function comparison')
plt.xlabel('Input z'); plt.ylabel('Activation output')
plt.grid(True); plt.legend(); plt.show()

Input values: [-2. -1.  0.  1.  2.]
Step: [0 0 1 1 1]
Sigmoid: [0.1192 0.2689 0.5    0.7311 0.8808]
ReLU: [0. 0. 0. 1. 2.]


## 5. Result and conclusion
- The perceptron learned the linearly separable loan-approval training data and reached **100% training accuracy** in the recorded run.
- The decision-boundary plot shows how the two input features are separated by a straight line.
- The single-layer perceptron did not converge to a correct XOR solution; this demonstrates its limitation on non-linearly separable data.
- Step, sigmoid and ReLU produce different output ranges and behaviours.

**Conclusion:** A single perceptron is suitable for linearly separable binary classification. A hidden layer or another non-linear model is required for patterns such as XOR.

*Note: The output text above is saved in the notebook for quick viewing. Run the cells in Google Colab to regenerate the plots and verify the results.*